# Generate Spectral Entropy Across Layers

This Generate notebook computes per-layer spectral entropy and caches the corresponding effective active modes, $\exp(\mathrm{SE})$, consumed by `Visualize-diagnostics-across-layers.ipynb`. The companion Visualize notebook compares layer-specific graph-spectral diagnostics across all 28 layers at a fixed context length of $N = 2000$ tokens for the main numerical input families using Llama-3.2-3B, providing the across-layer comparison presented under **Robustness and Layerwise Consistency of Graph-Spectral Diagnostics** in the appendix.


Set imports, constants, fixed-context configuration, and random seeds.

In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Seed data generation and model inference
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Fixed context: 1,000 numeric states, each followed by a comma
CONTEXT_STATE_TOKENS = 1000
CONTEXT_COMMA_TOKENS = 1000
FIXED_CONTENT_TOKENS = 2000
TOTAL_SEQUENCE_STATES = 1001

MODEL_NAME = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


In [ ]:
# Reference Lorenz trajectory used to define the input-family convention
def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]

sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0

initial_conditions = [-8, 7, 27]
lorenz_dt = 100 / 999
t_eval = np.arange(TOTAL_SEQUENCE_STATES) * lorenz_dt
t_span = (float(t_eval[0]), float(t_eval[-1]))

sol = solve_ivp(lorenz, t_span, initial_conditions, args=(sigma, rho, beta), t_eval=t_eval)


In [ ]:
# Quantize the Lorenz x-coordinate to the shared three-digit token range
x_vals = sol.y[0]
x_min = x_vals.min()
x_max = x_vals.max()

x_integers = np.round(50 + (x_vals - x_min) / (x_max - x_min) * 900).astype(int)


In [ ]:
# Reference input families used to define the fixed cache order
N = TOTAL_SEQUENCE_STATES

# Logistic-map regimes: six periodic windows plus two chaotic controls
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_rs = list(periodic_logistic_periods.keys()) + [3.9, 4.0]
logistic_expected_periods = {r: periodic_logistic_periods.get(r) for r in logistic_rs}
x0 = 0.2
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(n, dtype=float)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


def quantize_logistic_window(vals):
    vals = np.asarray(vals, dtype=float)
    vmin, vmax = vals.min(), vals.max()
    if vmax == vmin:
        return np.full(vals.shape[0], 500, dtype=int)
    return np.round(50 + (vals - vmin) / (vmax - vmin) * 900).astype(int)


def build_burned_in_logistic_window(r, x0, n_window, expected_period=None, n_long=20000, tol=1e-10):
    if expected_period is None:
        return logistic_sequence(r, x0, n_window)

    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")
    if t_settle + n_window > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; settle index {t_settle} leaves fewer than {n_window} states."
        )
    return np.asarray(x_long[t_settle:t_settle + n_window], dtype=float)


logistic_integers = {}
for r in logistic_rs:
    vals = build_burned_in_logistic_window(
        r,
        x0=x0,
        n_window=TOTAL_SEQUENCE_STATES,
        expected_period=logistic_expected_periods[r],
        n_long=logistic_long_n,
        tol=logistic_settle_tol,
    )
    logistic_integers[r] = quantize_logistic_window(vals)

input_data_types = {
    "Logistic r=3.2": logistic_integers[3.2],
    "Logistic r=3.5": logistic_integers[3.5],
    "Logistic r=3.55": logistic_integers[3.55],
    "Logistic r=3.566": logistic_integers[3.566],
    "Logistic r=3.5692": logistic_integers[3.5692],
    "Logistic r=3.5698": logistic_integers[3.5698],
    "Logistic r=3.9": logistic_integers[3.9],
    "Logistic r=4.0": logistic_integers[4.0],
    "Lorenz": x_integers,
}



Compute spectral entropy from same-index attention and hidden-state layers after token alignment.


In [ ]:
# Token alignment, graph construction, and spectral-entropy utilities
def get_token_start_index(input_ids, bos_token_id=None, drop_bos=True):
    if not drop_bos:
        return 0
    if input_ids is None or bos_token_id is None:
        return 0

    if isinstance(input_ids, torch.Tensor):
        first_id = int(input_ids.flatten()[0].item())
    else:
        first_id = int(np.asarray(input_ids).flatten()[0])

    return 1 if first_id == int(bos_token_id) else 0


def _resolve_hidden_state_layer_index(hidden_states, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(hidden_states) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(hidden_states):
        raise ValueError(
            f"Invalid hidden-state layer_index={layer_index} for {len(hidden_states)} hidden-state tensors"
        )
    return int(resolved_layer_index)


def _resolve_attention_layer_index(attentions, layer_index):
    resolved_layer_index = layer_index if layer_index >= 0 else len(attentions) + layer_index
    if resolved_layer_index < 0 or resolved_layer_index >= len(attentions):
        raise ValueError(
            f"Invalid attention layer_index={layer_index} for {len(attentions)} attention tensors"
        )
    return int(resolved_layer_index)


def compute_spectral_entropy_from_attn_and_hidden(
    attentions,
    hidden_states,
    input_ids=None,
    bos_token_id=None,
    head_weighting="mass",
    drop_bos=True,
    renormalize_after_drop=True,
    attention_layer_index=-1,
    signal_layer_index=-2,
):
    """Compute spectral entropy for one attention/hidden-state layer."""
    token_start = get_token_start_index(
        input_ids,
        bos_token_id=bos_token_id,
        drop_bos=drop_bos,
    )

    resolved_attention_layer_index = _resolve_attention_layer_index(attentions, attention_layer_index)
    resolved_signal_layer_index = _resolve_hidden_state_layer_index(hidden_states, signal_layer_index)

    # Remove the BOS token before forming the token graph used by the diagnostic
    A = attentions[resolved_attention_layer_index][0].float()
    A = A[:, token_start:, token_start:]
    seq_len = int(A.shape[-1])
    if seq_len <= 0:
        return {
            "spectral_entropy": float("nan"),
            "seq_len": seq_len,
            "token_start": int(token_start),
            "resolved_attention_layer_index": int(resolved_attention_layer_index),
            "resolved_signal_layer_index": int(resolved_signal_layer_index),
        }

    if renormalize_after_drop and token_start > 0:
        row_sums = A.sum(dim=-1, keepdim=True)
        row_sums = torch.where(row_sums == 0, torch.ones_like(row_sums), row_sums)
        A = A / row_sums

    # Symmetrize attention heads and combine them into one weighted graph
    W_heads = 0.5 * (A + A.transpose(-1, -2))
    if head_weighting == "uniform":
        alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
    elif head_weighting == "mass":
        s = A.sum(dim=(1, 2))
        s_sum = s.sum()
        if s_sum <= 0:
            alpha = torch.full((A.shape[0],), 1.0 / A.shape[0], device=A.device, dtype=A.dtype)
        else:
            alpha = s / s_sum
    else:
        raise ValueError("head_weighting must be 'mass' or 'uniform'")

    Wbar = (alpha[:, None, None] * W_heads).sum(dim=0)
    D = torch.diag(Wbar.sum(dim=1))
    L = D - Wbar
    _, U = torch.linalg.eigh(L)

    X = hidden_states[resolved_signal_layer_index][0].float()
    X = X[token_start:, :].to(U.device)
    if X.shape[0] != seq_len:
        raise ValueError(
            f"Signal/token length mismatch after token alignment: X has {X.shape[0]} rows, graph has {seq_len} nodes."
        )

    # Project hidden states onto graph-Laplacian eigenvectors and summarize the energy distribution
    X_hat = U.T @ X
    energy = (X_hat ** 2).sum(dim=1)
    total = energy.sum().item()
    if total == 0:
        spectral_entropy = float("nan")
    else:
        p = energy / total
        p = p[p > 0]
        spectral_entropy = float(-(p * torch.log(p)).sum().item())

    return {
        "spectral_entropy": spectral_entropy,
        "seq_len": int(seq_len),
        "token_start": int(token_start),
        "resolved_attention_layer_index": int(resolved_attention_layer_index),
        "resolved_signal_layer_index": int(resolved_signal_layer_index),
    }


spectral_entropy_fixed_state_tokens = CONTEXT_STATE_TOKENS
spectral_entropy_fixed_comma_tokens = CONTEXT_COMMA_TOKENS
spectral_entropy_fixed_content_tokens = FIXED_CONTENT_TOKENS
spectral_entropy_drop_bos = True
spectral_entropy_renormalize_after_drop = True
# Mass and uniform weighting are equivalent for the models studied here; mass also supports sparse-attention variants
spectral_entropy_head_weighting = "mass"
spectral_entropy_tokenization_add_special_tokens = True
spectral_entropy_comma_token_ids = tokenizer(",", add_special_tokens=False).input_ids
if len(spectral_entropy_comma_token_ids) != 1:
    raise ValueError(f"Expected comma to tokenize as one token, got {spectral_entropy_comma_token_ids}.")
spectral_entropy_comma_token_id = int(spectral_entropy_comma_token_ids[0])


def _build_fixed_content_token_sequence(
    sequence,
    n_state_tokens=spectral_entropy_fixed_state_tokens,
    n_comma_tokens=spectral_entropy_fixed_comma_tokens,
):
    """Build the fixed 2000-token prompt: 1000 states, each followed by a comma."""
    sequence = np.asarray(sequence, dtype=int)
    n_state_tokens = int(n_state_tokens)
    n_comma_tokens = int(n_comma_tokens)
    if n_state_tokens != n_comma_tokens:
        raise ValueError("Fixed-context diagnostic expects one comma after each numeric state token.")
    if sequence.shape[0] < TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Need at least {TOTAL_SEQUENCE_STATES} numeric states so state "
            f"{CONTEXT_STATE_TOKENS} is held out, got {sequence.shape[0]}."
        )

    context_window = sequence[:n_state_tokens]
    input_sequence = ",".join([f"{x:03d}" for x in context_window]) + ","
    return input_sequence, n_state_tokens, n_comma_tokens



Build the fixed-length ensemble input suite used by the per-layer sweep.


In [ ]:
# Share each replicate's sampled initial condition across input families
spectral_entropy_ensemble_R = 20
spectral_entropy_ensemble_seed = 42
spectral_entropy_periodic_random_phase = True

spectral_entropy_ensemble_rng = np.random.default_rng(spectral_entropy_ensemble_seed)
spectral_entropy_x0_samples = spectral_entropy_ensemble_rng.uniform(0.3, 0.7, size=spectral_entropy_ensemble_R)
spectral_entropy_ensemble_rep_ids = list(range(spectral_entropy_ensemble_R))

spectral_entropy_ensemble_periodic_rs = [3.2, 3.5, 3.55, 3.566, 3.5692, 3.5698]
spectral_entropy_ensemble_chaotic_rs = [3.9, 4.0]
spectral_entropy_ensemble_data_order = ["Constant sampled"] + list(input_data_types.keys())



In [ ]:
def build_burned_in_logistic_window_ensemble(
    r,
    x0,
    n_window,
    expected_period=None,
    n_long=20000,
    tol=1e-10,
    phase_offset=0,
):
    if expected_period is None:
        return np.asarray(logistic_sequence(r, x0, n_window), dtype=float)

    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(
            f"Could not find settle time for r={r} with expected period {expected_period}."
        )

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(
            f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states."
        )

    return np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)


def quantize_lorenz_x_values(x_values):
    x_values = np.asarray(x_values, dtype=float)
    x_min_rep = float(np.min(x_values))
    x_max_rep = float(np.max(x_values))
    if x_max_rep == x_min_rep:
        return np.full(x_values.shape[0], 500, dtype=int)
    return np.round(50 + (x_values - x_min_rep) / (x_max_rep - x_min_rep) * 900).astype(int)


spectral_entropy_ensemble_phase_rng = np.random.default_rng(spectral_entropy_ensemble_seed + 1)
spectral_entropy_ensemble_inputs = {}

for rep_id, x0_rep in zip(spectral_entropy_ensemble_rep_ids, spectral_entropy_x0_samples):
    x0_rep = float(x0_rep)
    rep_inputs = {}

    constant_token = int(np.round(50 + 900 * x0_rep))
    rep_inputs["Constant sampled"] = {
        "sequence": np.full(TOTAL_SEQUENCE_STATES, constant_token, dtype=int),
    }

    for r in spectral_entropy_ensemble_periodic_rs:
        expected_period = logistic_expected_periods[r]
        phase_offset = (
            int(spectral_entropy_ensemble_phase_rng.integers(0, expected_period))
            if spectral_entropy_periodic_random_phase
            else 0
        )
        vals = build_burned_in_logistic_window_ensemble(
            r,
            x0=x0_rep,
            n_window=TOTAL_SEQUENCE_STATES,
            expected_period=expected_period,
            n_long=logistic_long_n,
            tol=logistic_settle_tol,
            phase_offset=phase_offset,
        )
        rep_inputs[f"Logistic r={r}"] = {
            "sequence": quantize_logistic_window(vals),
        }

    for r in spectral_entropy_ensemble_chaotic_rs:
        vals = logistic_sequence(r, x0_rep, TOTAL_SEQUENCE_STATES)
        rep_inputs[f"Logistic r={r}"] = {
            "sequence": quantize_logistic_window(vals),
        }

    sol_rep = solve_ivp(
        lorenz,
        t_span,
        [float(-8.0 + (x0_rep - 0.5)), 7.0, 27.0],
        args=(sigma, rho, beta),
        t_eval=t_eval,
    )
    if not sol_rep.success:
        raise RuntimeError(f"Lorenz solve failed for rep_id={rep_id}: {sol_rep.message}")
    if sol_rep.y.shape[-1] != TOTAL_SEQUENCE_STATES:
        raise ValueError(
            f"Expected {TOTAL_SEQUENCE_STATES} Lorenz states for rep_id={rep_id}, "
            f"got {sol_rep.y.shape[-1]}."
        )
    rep_inputs["Lorenz"] = {
        "sequence": quantize_lorenz_x_values(sol_rep.y[0]),
    }


    spectral_entropy_ensemble_inputs[int(rep_id)] = rep_inputs



Load the model with eager attention and capture layer/device metadata.


In [ ]:
# Use eager attention to return full attention tensors
model_attn = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager",
)
model_attn.eval()

if int(model_attn.config.num_hidden_layers) != 28:
    raise AssertionError(f"Expected model to have 28 layers, got {model_attn.config.num_hidden_layers}.")


Run the full per-layer spectral-entropy sweep for each input family and realization.


In [ ]:
spectral_entropy_ensemble_across_layer_records = []
device = next(model_attn.parameters()).device
spectral_entropy_num_layers = int(model_attn.config.num_hidden_layers)
spectral_entropy_expected_shape = (10, 20, 28)
# Task grid: realization x input family
spectral_entropy_ensemble_tasks = [
    (int(rep_id), data_name)
    for rep_id in spectral_entropy_ensemble_rep_ids
    for data_name in spectral_entropy_ensemble_data_order
]


for rep_id, data_name in tqdm(spectral_entropy_ensemble_tasks, desc="Ensemble per-layer spectral entropy"):
    entry = spectral_entropy_ensemble_inputs[int(rep_id)][data_name]
    sequence = np.asarray(entry["sequence"], dtype=int)

    input_sequence, n_state_tokens, n_comma_tokens = _build_fixed_content_token_sequence(sequence)
    inputs = tokenizer(
        input_sequence,
        add_special_tokens=spectral_entropy_tokenization_add_special_tokens,
        return_tensors="pt",
    ).to(device)

    # Validate post-BOS prompt length before the expensive model pass
    token_start = get_token_start_index(
        inputs.input_ids[0],
        bos_token_id=tokenizer.bos_token_id,
        drop_bos=spectral_entropy_drop_bos,
    )
    post_bos_prompt_token_ids = inputs.input_ids[0, token_start:].detach().cpu().tolist()
    post_bos_token_dim = len(post_bos_prompt_token_ids)
    if post_bos_token_dim != spectral_entropy_fixed_content_tokens:
        raise ValueError(
            f"Expected {spectral_entropy_fixed_content_tokens} content tokens, "
            f"got {post_bos_token_dim} for rep_id={rep_id}, data_name={data_name}."
        )
    if int(post_bos_prompt_token_ids[-1]) != spectral_entropy_comma_token_id:
        raise ValueError(
            f"Expected final post-BOS token to be comma token {spectral_entropy_comma_token_id}, "
            f"got {post_bos_prompt_token_ids[-1]} for rep_id={rep_id}, data_name={data_name}."
        )

    # Request attention and hidden-state tensors for all model layers
    with torch.no_grad():
        outputs = model_attn(
            **inputs,
            output_attentions=True,
            output_hidden_states=True,
            return_dict=True,
        )

    if len(outputs.attentions) != spectral_entropy_num_layers:
        raise ValueError(
            f"Expected {spectral_entropy_num_layers} attention tensors, got {len(outputs.attentions)}."
        )
    if len(outputs.hidden_states) != spectral_entropy_num_layers + 1:
        raise ValueError(
            f"Expected {spectral_entropy_num_layers + 1} hidden-state tensors, got {len(outputs.hidden_states)}."
        )

    # Pair each attention layer with the same-index hidden-state layer
    for layer_idx in range(spectral_entropy_num_layers):
        diagnostic_bundle = compute_spectral_entropy_from_attn_and_hidden(
            outputs.attentions,
            outputs.hidden_states,
            input_ids=inputs.input_ids[0],
            bos_token_id=tokenizer.bos_token_id,
            head_weighting=spectral_entropy_head_weighting,
            drop_bos=spectral_entropy_drop_bos,
            renormalize_after_drop=spectral_entropy_renormalize_after_drop,
            attention_layer_index=layer_idx,
            signal_layer_index=layer_idx,
        )

        if int(diagnostic_bundle["seq_len"]) != spectral_entropy_fixed_content_tokens:
            raise ValueError(
                f"Layer {layer_idx + 1}: expected graph/token dimension "
                f"{spectral_entropy_fixed_content_tokens}, got {diagnostic_bundle['seq_len']}."
            )
        if (
            int(diagnostic_bundle["resolved_attention_layer_index"]) != layer_idx
            or int(diagnostic_bundle["resolved_signal_layer_index"]) != layer_idx
        ):
            raise ValueError(
                f"Layer-pairing check failed for layer {layer_idx}: "
                f"attention={diagnostic_bundle['resolved_attention_layer_index']}, "
                f"signal={diagnostic_bundle['resolved_signal_layer_index']}."
            )

        spectral_entropy_ensemble_across_layer_records.append(
            {
                "rep_id": int(rep_id),
                "data_name": data_name,
                "layer_index": int(layer_idx),
                "layer_number": int(layer_idx + 1),
                "attention_layer_index": int(diagnostic_bundle["resolved_attention_layer_index"]),
                "signal_layer_index": int(diagnostic_bundle["resolved_signal_layer_index"]),
                "spectral_entropy": float(diagnostic_bundle["spectral_entropy"]),
            }
        )

    del outputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

spectral_entropy_ensemble_across_layer_df = pd.DataFrame(spectral_entropy_ensemble_across_layer_records)
spectral_entropy_ensemble_across_layer_df["data_name"] = pd.Categorical(
    spectral_entropy_ensemble_across_layer_df["data_name"],
    categories=spectral_entropy_ensemble_data_order,
    ordered=True,
)
spectral_entropy_ensemble_across_layer_df = spectral_entropy_ensemble_across_layer_df.sort_values(
    ["data_name", "rep_id", "layer_number"]
).reset_index(drop=True)

spectral_entropy_diagnostic_shape = (
    int(spectral_entropy_ensemble_across_layer_df["data_name"].nunique()),
    int(spectral_entropy_ensemble_across_layer_df["rep_id"].nunique()),
    int(spectral_entropy_ensemble_across_layer_df["layer_number"].nunique()),
)
if spectral_entropy_diagnostic_shape != spectral_entropy_expected_shape:
    raise ValueError(
        f"Diagnostic shape sanity check failed: observed {spectral_entropy_diagnostic_shape}, "
        f"expected {spectral_entropy_expected_shape}."
    )



Summarize effective active modes across realizations and save the visualization cache.


In [ ]:
# Keep cache paths relative to this notebook's working directory
spectral_entropy_cache_dir = Path("cache")
spectral_entropy_cache_dir.mkdir(exist_ok=True)
spectral_entropy_output_path = spectral_entropy_cache_dir / "spectral_entropy_per_layer.npz"

spectral_entropy_input_names = np.asarray(spectral_entropy_ensemble_data_order, dtype=str)
spectral_entropy_run_ids = np.asarray(spectral_entropy_ensemble_rep_ids, dtype=np.int64)
spectral_entropy_layer_numbers = np.arange(1, spectral_entropy_num_layers + 1, dtype=np.int64)


def _spectral_entropy_array_from_records():
    diagnostic_array = np.full(
        (len(spectral_entropy_input_names), len(spectral_entropy_run_ids), spectral_entropy_num_layers),
        np.nan,
        dtype=np.float64,
    )
    for input_idx, data_name in enumerate(spectral_entropy_ensemble_data_order):
        for run_idx, rep_id in enumerate(spectral_entropy_ensemble_rep_ids):
            df_diagnostic = spectral_entropy_ensemble_across_layer_df[
                (spectral_entropy_ensemble_across_layer_df["data_name"] == data_name)
                & (spectral_entropy_ensemble_across_layer_df["rep_id"] == rep_id)
            ].sort_values("layer_index")
            if len(df_diagnostic) != spectral_entropy_num_layers:
                raise ValueError(
                    f"Expected {spectral_entropy_num_layers} layers for {data_name}, rep_id={rep_id}; "
                    f"got {len(df_diagnostic)}."
                )
            diagnostic_array[input_idx, run_idx, :] = df_diagnostic["spectral_entropy"].to_numpy(dtype=np.float64)
    return diagnostic_array


spectral_entropy_per_run = _spectral_entropy_array_from_records()
expected_spectral_entropy_shape = (10, 20, 28)
if spectral_entropy_per_run.shape != expected_spectral_entropy_shape:
    raise ValueError(
        f"spectral_entropy_per_run shape check failed: observed {spectral_entropy_per_run.shape}, "
        f"expected {expected_spectral_entropy_shape}."
    )
if np.isnan(spectral_entropy_per_run).any():
    raise ValueError("spectral_entropy_per_run contains NaNs.")

# Convert spectral entropy to effective active modes and summarize across realizations
effective_modes_per_run = np.exp(spectral_entropy_per_run)
effective_modes_mean = np.nanmean(effective_modes_per_run, axis=1)
if len(spectral_entropy_run_ids) > 1:
    effective_modes_std = np.nanstd(effective_modes_per_run, axis=1, ddof=1)
else:
    effective_modes_std = np.zeros_like(effective_modes_mean)

expected_summary_shape = (len(spectral_entropy_input_names), spectral_entropy_num_layers)
for array_name, array_value in {
    "effective_modes_mean": effective_modes_mean,
    "effective_modes_std": effective_modes_std,
}.items():
    if array_value.shape != expected_summary_shape:
        raise ValueError(
            f"{array_name} shape check failed: observed {array_value.shape}, "
            f"expected {expected_summary_shape}."
        )

# Save compact arrays for the companion Visualize notebook
np.savez_compressed(
    spectral_entropy_output_path,
    input_names=spectral_entropy_input_names,
    layer_numbers=spectral_entropy_layer_numbers,
    effective_modes_mean=effective_modes_mean,
    effective_modes_std=effective_modes_std,
)
